# Evergreen Python 백테스트 노트북

이 노트북은 Python 백테스트의 README이자 스모크 테스트다. 아래 설정 셀에서 버전, 데이터 소스, 시장, 탐색 개수만 바꾸면 같은 방식으로 v1부터 v5까지 비교 실행할 수 있다.

기본 데이터 소스는 공식 Upbit SDK를 쓰는 `upbit`이다. 네트워크 없이 구조만 확인하려면 `데이터_소스 = "synthetic"`으로 바꾸면 되고, 실제 Upbit 일봉 데이터는 `outputs/data/upbit-cache`에 CSV로 캐시된다.

현재 공식 SDK는 캔들/주문/계좌 같은 OpenAPI 리소스를 제공하고, RSI/EMA/ATR 같은 전략 지표 계산기는 제공하지 않는다. 그래서 데이터 조회는 SDK를 쓰고, 지표 계산은 버전별 전략 코드 안에서 재현 가능하게 유지한다.

## 전략 설명서

### 공통 연동정의

모든 전략은 같은 입력과 출력 계약을 따른다.

- 입력: `CandleBar[]`, `signal_index`, 현재 포지션, 버전별 파라미터 JSON
- 출력: `StrategyEvaluation`
- 실행 판단: `decision.action`은 `BUY`, `SELL`, `HOLD` 중 하나
- 주문 비중: `decision.targetPositionRatio`는 목표 포지션 비중이다. 0.0은 전량 현금, 1.0은 전량 보유다. v3처럼 변동성 비중 전략은 `max_leverage`까지 값을 낼 수 있으며, 1.0보다 크면 레버리지 노출을 의미한다.
- 판단 이유: `decision.signalReason`은 매수/매도/유지의 원인을 기록한다.
- 진단값: `diagnostics`는 전략별 보조 지표다. 실행 판단의 필수 계약이 아니라 로그, 차트, 디버깅용이다.

Java에서 전략을 연동할 때는 `action`만 보지 말고 `targetPositionRatio`까지 함께 읽어야 한다. 특히 v3는 매수/매도 신호가 전체 진입/청산이 아니라 목표 비중 조정일 수 있다.

현재 Java 주문 계층은 `targetPositionRatio`가 0.0 또는 1.0일 때만 실제 주문으로 실행하고, 그 외의 중간 비중은 로그를 남기고 스킵한다. 따라서 v3 같은 비중 조절 전략은 계약/관찰 후보로는 실행 가능하지만, 운영 주문까지 쓰려면 Java 주문 계층의 목표 비중 실행을 먼저 구현해야 한다.

### v1: MA + RSI

가장 단순한 추세 추종 베이스라인이다. 가격이 이동평균 위에 있고 이동평균 기울기가 상승이며 RSI가 설정값보다 낮을 때 매수하고, 가격이 이동평균 아래로 내려가면 매도한다. 포지션은 0 또는 1의 전량 진입/전량 청산이다. 주요 조정값은 `rsi_buy`, `ma_len`, `ma_slope_days`다.

### v2: Regime + ATR Stop

EMA 밴드로 상승/하락 국면을 나누고 ATR 추적 손절을 붙인 전략이다. `BEAR`에서 `BULL`로 전환될 때 매수하고, `BULL`에서 `BEAR`로 전환되거나 ATR 추적 손절이 발생하면 매도한다. 주요 조정값은 `regime_ema_len`, `atr_period`, `atr_trail_multiplier`, `regime_band`다.

### v3: Volatility Target + Regime

v2 구조에 변동성 기반 목표 비중 조절을 추가한 전략이다. 상승 국면에서 목표 변동성에 맞춰 현재 비중보다 목표 비중이 커지면 증액하고, 하락 국면이나 ATR 추적 손절 또는 목표 비중 축소가 필요하면 감액한다. 포지션은 0.0부터 `max_leverage`까지의 동적 목표 비중이다. 주요 조정값은 `vol_target`, `max_leverage`, `min_exposure`와 레짐/ATR 파라미터다.

### v4: Weekly Filter

v2에 주간 EMA 필터를 추가한 전략이다. 일봉 레짐이 매수 신호를 내고 주간 필터도 상승으로 판단할 때만 진입한다. 매도는 일봉 레짐 매도 또는 ATR 추적 손절 기준을 따른다. 주요 조정값은 `weekly_ema_len`과 v2 계열 파라미터다.

### v5: Adaptive ATR Exit

v2의 ATR 추적 손절을 변동성 국면에 따라 다르게 적용하는 전략이다. `BEAR`에서 `BULL`로 전환될 때 매수하고, `BULL`에서 `BEAR`로 전환되거나 현재 변동성 국면에 맞춘 ATR 추적 손절이 발생하면 매도한다. 주요 조정값은 `atr_mult_low_vol`, `atr_mult_high_vol`, `vol_regime_lookback`, `vol_regime_threshold`, `regime_band`다.

### 선택 기준

- 단순 기준선이 필요하면 v1
- 안정적인 레짐/손절 베이스라인이 필요하면 v2
- 비중 조절까지 실험하려면 v3
- 큰 추세 필터를 강하게 걸고 싶으면 v4
- 현재 Java 엔진과 가장 가까운 운영 후보는 v5

## 1. 프로젝트 경로와 커널 확인

IntelliJ나 Jupyter를 어디서 열어도 import가 되도록 프로젝트 루트를 찾고 `PYTHONPATH`에 추가한다. 다른 위치에서 실행해야 하면 환경 변수 `EVERGREEN_PROJECT_ROOT`에 프로젝트 루트를 넣으면 된다.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def 프로젝트_루트_찾기() -> Path:
    env_root = os.environ.get("EVERGREEN_PROJECT_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    candidates = [Path.cwd(), *Path.cwd().parents, Path("/Users/moon/IdeaProjects/evergreen")]
    for candidate in candidates:
        if (candidate / "settings.gradle").exists() and (candidate / "evergreen_backtest/strategies/v5.py").exists():
            return candidate.resolve()
    raise RuntimeError("Evergreen 프로젝트 루트를 찾지 못했다. EVERGREEN_PROJECT_ROOT를 설정해라.")


PROJECT_ROOT = 프로젝트_루트_찾기()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs" / ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / "outputs" / ".cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

for candidate in (PROJECT_ROOT,):
    text = str(candidate)
    if text not in sys.path:
        sys.path.insert(0, text)

print("프로젝트 루트:", PROJECT_ROOT)
print("Python:", sys.executable)

## 2. 실행 API 불러오기

전략 버전은 `evergreen_backtest/strategies/v*.py`에서 자동으로 찾고, 실행은 `evergreen_backtest` 패키지가 담당한다.

In [ ]:
from datetime import datetime, timezone
from pprint import pprint

from evergreen_backtest import BacktestRunRequest, available_versions, run_backtest
from evergreen_backtest.plotting import plot_equity_curves

print("사용 가능한 전략 버전:", ", ".join(available_versions()))

## 3. 여기만 바꿔서 실행

- `실행할_버전`: `("v5",)`처럼 하나만 넣거나 `("v3", "v4", "v5")`처럼 여러 개를 넣는다.
- `데이터_소스`: `synthetic`, `cache`, `upbit`, `sdk` 중 하나를 쓴다.
- `공통_설정`: 모든 버전에 적용한다. 각 버전이 모르는 항목은 자동으로 무시된다.
- `버전별_설정`: 특정 버전에만 적용한다. 오타나 지원하지 않는 항목은 바로 에러를 낸다.

In [ ]:
실행할_버전 = ("v3", "v4", "v5")
데이터_소스 = "upbit"  # "upbit" | "sdk" | "cache" | "synthetic"
마켓 = "KRW-BTC"
시작일 = datetime(2020, 1, 1, tzinfo=timezone.utc)
종료일 = None
합성_데이터_개수 = 420

공통_설정 = {
    "top_k": 5,
    "grid_parallelism": 1,
}

버전별_설정 = {
    "v5": {
        # 예: "atr_window": 14,
    },
}

## 4. 백테스트 실행

이 셀이 노트북의 테스트 역할을 한다. 실행 결과가 비어 있거나 요청한 버전이 빠지면 바로 실패한다.

In [ ]:
요청 = BacktestRunRequest(
    versions=실행할_버전,
    source=데이터_소스,
    market=마켓,
    from_dt=시작일,
    to_dt=종료일,
    cache_dir=PROJECT_ROOT / "outputs" / "data" / "upbit-cache",
    synthetic_count=합성_데이터_개수,
    common_config=공통_설정,
    version_config=버전별_설정,
    output_dir=PROJECT_ROOT / "outputs" / "backtests" / "latest",
)

결과 = run_backtest(요청)
요약 = 결과.summary_dataframe()

assert not 요약.empty, "백테스트 요약이 비어 있다."
assert set(실행할_버전).issubset(set(요약["version"])), "요청한 버전 중 누락된 결과가 있다."

요약

## 5. 자산 곡선 보기

`phase`는 `validation`, `test`, `full` 중 하나를 넣을 수 있다. 기본 비교 구간은 최적화 이후 검증용으로 쓰는 `test`다.

In [ ]:
그림 = plot_equity_curves(
    결과,
    phase="test",
    save_to=PROJECT_ROOT / "outputs" / "backtests" / "latest" / "equity_test.png",
)
그림

## 6. Java에서 바로 쓰는 계약 파일

`strategy_contracts.json`에는 계약 스키마 버전, 공통 StrategyInput/StrategyEvaluation 정의, 버전별 선택 파라미터, Java 파라미터 클래스명, 마지막 평가의 `decision.action`(`BUY`/`SELL`/`HOLD`), `decision.signalReason`, `decision.targetPositionRatio`, 검증/테스트/전체 성과 요약이 들어간다. `ioContractReady`는 v1~v5가 같은 연동정의를 만족한다는 뜻이고, `javaEngineReady`는 해당 버전의 Java 엔진이 실제 구현되어 있다는 뜻이다. Java 쪽에서는 `javaParamsCamelCase`를 DTO나 설정 객체에 매핑하면 된다. `selectedParamsCamelCase`는 Python 백테스트의 전체 파라미터를 보존한다. 현재 Java 주문 실행은 `targetPositionRatio` 0.0과 1.0만 지원하므로, 중간 비중 전략은 계약 파일로 먼저 확인한 뒤 주문 계층을 확장해야 한다.

In [ ]:
출력_경로 = 결과.write_outputs()
계약 = 결과.contracts()

print(f"출력 경로: {출력_경로}")
print("\nJava 전달용 파라미터와 마지막 평가 예시")
for 버전, payload in 계약["versions"].items():
    java_class = payload["javaParameterClass"] or "아직 Java 엔진 구현 없음"
    print("-", 버전, java_class)
    print("  공통 I/O 계약:", payload["ioContractReady"], "Java 엔진:", payload["javaEngineReady"])
    if payload["javaEngineReady"]:
        pprint(payload["javaParamsCamelCase"])
    else:
        print("  Python 백테스트 파라미터")
        pprint(payload["selectedParamsCamelCase"])
    decision = payload["lastEvaluation"]["decision"]
    print("  마지막 평가:", decision["action"], decision["signalReason"], "target=", decision["targetPositionRatio"])

## 7. 실제 Upbit 데이터로 실행할 때

공식 SDK를 쓰려면 의존성을 동기화한 뒤 `uv (evergreen)` 커널로 열고 `데이터_소스 = "upbit"`으로 바꿔서 다시 실행한다.

```bash
uv sync
uv run jupyter lab backtest_playground.ipynb
```

최초 실행은 Upbit 일봉 API를 호출하고, 같은 기간/마켓은 다음부터 CSV 캐시를 먼저 읽는다. API 키가 필요한 주문/계좌 기능은 여기서 사용하지 않고, 공개 캔들 데이터만 조회한다.